In [8]:
%pip install qiskit==1.2.4
%pip install qiskit-aer==0.15.1
%pip install pylatexenc==2.10

from qiskit import QuantumCircuit
from qiskit.converters import circuit_to_gate
from qiskit.visualization import array_to_latex
from qiskit.quantum_info import Operator
from qiskit.quantum_info import Statevector
from qiskit import transpile
from qiskit.providers.basic_provider import BasicSimulator
from qiskit.visualization import plot_histogram
from qiskit.circuit import ControlledGate
import math

In [9]:
# The aim of the assignment is to simulate the BB84 key distribution protocol.

# This notebook is for a simulation of the protocol without an attacker.



In [10]:
def qrng(n):
    backend = BasicSimulator()
    bits = []
    while len(bits) < n:
        batch = min(20, n - len(bits))
        qc = QuantumCircuit(batch, batch)
        for i in range(batch):
            qc.h(i)
        qc.measure(range(batch), range(batch))
        job = backend.run(transpile(qc, backend), shots=1)
        counts = job.result().get_counts()
        result = list(counts.keys())[0][::-1]
        bits += [int(b) for b in result]
    return bits

N = 20

print("testing qrng:", qrng(8))

testing qrng: [0, 1, 0, 1, 1, 0, 1, 0]


In [11]:
alice_bits = qrng(N)
alice_bases = qrng(N)

print("Alice's bits: ", alice_bits)
print("Alice's bases:", alice_bases)

Alice's bits:  [0, 0, 1, 0, 0, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 0, 1, 1, 0, 1]
Alice's bases: [1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 0, 1, 0, 1, 0, 0, 1, 1]


In [12]:
bob_bases = qrng(N)

backend = BasicSimulator()
bob_bits = []

for i in range(N):
    qc = QuantumCircuit(1, 1)
    if alice_bits[i] == 1:
        qc.x(0)
    if alice_bases[i] == 1:
        qc.h(0)
    if bob_bases[i] == 1:
        qc.h(0)
    qc.measure(0, 0)
    job = backend.run(transpile(qc, backend), shots=1)
    counts = job.result().get_counts()
    bob_bits.append(int(list(counts.keys())[0]))

print("Bob's bases:", bob_bases)
print("Bob's bits: ", bob_bits)

Bob's bases: [0, 0, 1, 1, 0, 0, 0, 0, 1, 1, 0, 1, 1, 0, 1, 1, 1, 0, 0, 1]
Bob's bits:  [0, 0, 1, 0, 0, 1, 1, 0, 1, 0, 1, 1, 0, 0, 1, 0, 1, 1, 1, 1]


In [13]:
alice_key = []
bob_key = []

for i in range(N):
    if alice_bases[i] == bob_bases[i]:
        alice_key.append(alice_bits[i])
        bob_key.append(bob_bits[i])

print("Alice's key:", alice_key)
print("Bob's key:  ", bob_key)
print("Key length:", len(alice_key), "out of", N, "qubits")

Alice's key: [0, 1, 0, 1, 0, 1, 1, 0, 1, 1]
Bob's key:   [0, 1, 0, 1, 0, 1, 1, 0, 1, 1]
Key length: 10 out of 20 qubits


In [14]:
errors = 0
for i in range(len(alice_key)):
    if alice_key[i] != bob_key[i]:
        errors += 1

print("Number of errors:", errors)
print("Keys match:", alice_key == bob_key)

Number of errors: 0
Keys match: True
